# 02. Regime EDA

설비 변경 이벤트를 기준으로 2018~2024 EMS reduced measurement의 분포 변화를 확인합니다.

In [ ]:
# C01. 환경 설정과 라이브러리
from pathlib import Path
import os
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import psycopg
from dotenv import load_dotenv
try:
    import koreanize_matplotlib  # noqa: F401
except Exception:
    pass
plt.rcParams['axes.unicode_minus'] = False
ROOT = Path.cwd()
FIG_DIR = ROOT / 'outputs/figures/regime_eda'
TAB_DIR = ROOT / 'outputs/tables/regime_eda'
FIG_DIR.mkdir(parents=True, exist_ok=True)
TAB_DIR.mkdir(parents=True, exist_ok=True)
load_dotenv()

In [ ]:
# C02. Regime 이벤트 정의
REGIMES = [
    ('R0_initial_operation', '초기 운영', '2017-12-30 23:00:00+00', '2019-02-01 00:00:00+00', 'PV 없음, CHP 초기 On/Off 운전'),
    ('R1_chp_logic_update', 'CHP 로직 변경 후', '2019-02-01 00:00:00+00', '2019-06-28 22:00:00+00', 'CHP 50~100% 모듈레이션 구간'),
    ('R2_pv_phase1', 'PV 1차 설치 후', '2019-06-28 22:00:00+00', '2020-03-01 00:00:00+00', 'PV 1차 설치 이후'),
    ('R3_covid', 'COVID 영향 구간', '2020-03-01 00:00:00+00', '2020-06-01 00:00:00+00', 'COVID 초기 부하 변화 구간'),
    ('R4_pv_phase2', 'PV 2차 증설 후', '2020-06-01 00:00:00+00', '2020-09-09 12:00:00+00', 'PV 풀용량 전환 이후'),
    ('R5_meter_replacement', '계량기 교체 이후', '2020-09-09 12:00:00+00', '2023-06-01 00:00:00+00', 'H2.Z35/H2.Z36 계량기 교체 이후 안정 구간'),
    ('R6_heating_modernization', '난방 현대화 이후', '2023-06-01 00:00:00+00', '2024-01-01 01:00:00+00', '난방 현대화 이후 데이터 종료 전 구간'),
]
EVENTS = [
    ('2019-02-01', 'CHP 로직 변경'),
    ('2019-06-28', 'PV 1차 설치'),
    ('2020-03-01', 'COVID 시작'),
    ('2020-06-01', 'PV 2차 증설'),
    ('2020-09-09', '계량기 교체'),
    ('2023-06-01', '난방 현대화'),
]
regime_df = pd.DataFrame(REGIMES, columns=['regime_code','regime_name','start_ts','end_ts','description'])
regime_df

In [ ]:
# C03. DB 연결 정보 구성
# .env에는 DB_HOST, DB_PORT, DB_NAME, DB_USER, DB_PASSWORD가 있어야 합니다.
conninfo = dict(
    host=os.environ['DB_HOST'],
    port=os.environ['DB_PORT'],
    dbname=os.environ['DB_NAME'],
    user=os.environ['DB_USER'],
    password=os.environ['DB_PASSWORD'],
)
{k: ('***' if k == 'password' else v) for k, v in conninfo.items()}

In [ ]:
# C04. reduced_measurement_1h 월별·regime별 집계
# PostgreSQL 세션 timeout은 해제합니다.
case_parts = []
for code, name, start, end, desc in REGIMES:
    case_parts.append(f"when ts >= timestamptz '{start}' and ts < timestamptz '{end}' then '{code}'")
regime_case = 'case ' + ' '.join(case_parts) + " else 'outside' end"
series_filter = """
    (category='electricity' and subcategory in ('total','pv','chp') and measurement='P')
    or (category='cooling' and subcategory='total' and measurement='P')
    or (category='heating' and subcategory='total' and measurement='P')
    or (category='weather' and subcategory='weather' and measurement in ('Ta','Igm'))
"""
with psycopg.connect(**conninfo) as conn:
    conn.execute('set statement_timeout=0')
    conn.execute('set lock_timeout=0')
    monthly = pd.read_sql(f"""
        select date_trunc('month', ts) as month,
               {regime_case} as regime_code,
               category, subcategory, measurement,
               count(*) as n_obs, avg(value) as mean_value,
               percentile_cont(0.1) within group (order by value) as p10,
               percentile_cont(0.5) within group (order by value) as p50,
               percentile_cont(0.9) within group (order by value) as p90
        from ems.reduced_measurement_1h
        where {series_filter}
        group by 1,2,3,4,5
        order by 1,3,4,5
    """, conn)
monthly['month'] = pd.to_datetime(monthly['month'])
monthly.head()

In [ ]:
# C05. 산출물 저장
monthly.to_csv(TAB_DIR / 'monthly_reduced_1h_regime.csv', index=False, encoding='utf-8-sig')
regime_df.to_csv(TAB_DIR / 'regime_events.csv', index=False, encoding='utf-8-sig')
monthly.shape

In [ ]:
# C06. 월별 평균 P와 regime 이벤트 시각화
label_map = {
    ('electricity','total','P'): '전기 전체 P',
    ('electricity','pv','P'): 'PV 발전 P',
    ('electricity','chp','P'): 'CHP 전기 P',
    ('cooling','total','P'): '냉방 전체 P',
    ('heating','total','P'): '난방 전체 P',
    ('weather','weather','Ta'): '외기온 Ta',
    ('weather','weather','Igm'): '일사량 Igm',
}
monthly['series'] = monthly.apply(lambda r: label_map.get((r['category'], r['subcategory'], r['measurement']), f"{r['category']}/{r['subcategory']}/{r['measurement']}"), axis=1)
fig, ax = plt.subplots(figsize=(14,7))
for series in ['전기 전체 P','냉방 전체 P','난방 전체 P']:
    d = monthly[monthly['series']==series].sort_values('month')
    ax.plot(d['month'], d['mean_value'], marker='o', markersize=2.5, linewidth=1.4, label=series)
for date,label in EVENTS:
    x = pd.to_datetime(date, utc=True).tz_localize(None)
    ax.axvline(x, color='gray', linestyle='--', linewidth=0.8, alpha=0.65)
    ax.text(x, ax.get_ylim()[1], label, rotation=90, va='top', ha='right', fontsize=8, color='dimgray')
ax.set_title('Regime 이벤트와 월별 평균 전력 P')
ax.set_xlabel('월')
ax.set_ylabel('월별 평균 P')
ax.legend()
ax.grid(alpha=0.25)
fig.tight_layout()
fig.savefig(FIG_DIR / 'monthly_power_regime_events.png', dpi=160)
plt.show()

## 1차 해석 메모

- 6년 전체를 단일 정상 분포로 보지 않고 설비 이벤트 기준으로 나누어 봅니다.
- 이 노트북은 모델 피처 확정이 아니라 regime 분할 필요성을 확인하는 EDA입니다.
- 다음 단계에서는 measurement 계산식과 계량기 balance/redundancy 관계를 별도 노트북에서 확인합니다.